In [2]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model_id = "Qwen/Qwen3-0.6B"
save_path = "./models/qwen3-0.6b"

# This downloads the files from Hugging Face
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id)

# This saves them to your local folder
tokenizer.save_pretrained(save_path)
model.save_pretrained(save_path)

print(f"Model saved locally to {save_path}")

Writing model shards: 100%|██████████| 1/1 [00:04<00:00,  4.60s/it]

Model saved locally to ./models/qwen3-0.6b


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

model_id = "Qwen/Qwen3-0.6B"

# 1. Load the tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_id)

# 2. Load the model on the CPU
# We use torch_dtype=torch.float32 because most CPUs handle it better than float16
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map={"": "cpu"}, 
    torch_dtype=torch.float32
)

# 3. Simple generation function
def chat_with_ai(prompt):
    inputs = tokenizer(prompt, return_tensors="pt")
    outputs = model.generate(**inputs, max_new_tokens=50)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)


`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 311/311 [00:01<00:00, 191.05it/s]


Write a JSON object for a user profile. The user has the following details: first name: John, last name: Smith, age: 32, gender: male, address: New York, city: New York, state: NY, country: USA, phone number: 5


In [4]:
prompt = "What is a good place to visit in Delhi?"
print(chat_with_ai(prompt))

What is a good place to visit in Delhi? What are the reasons for choosing a specific place? What are the attractions of each place? What are the challenges of choosing a place? What are the benefits of choosing a place? What are the possible alternative places to consider? What is the overall recommendation


In [7]:
import asyncio
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

# This defines how to talk to the Jupyter MCP server
server_params = StdioServerParameters(
    command="python",
    args=["-m", "jupyter_mcp_server"]
)

async def run_agentic_qwen():
    async with stdio_client(server_params) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            
            # 1. Use MCP to see what files are in your Strapi/React project
            tools = await session.list_tools()
            print(f"✅ MCP Connected. Tools available: {[t.name for t in tools.tools]}")
            
            # 2. Example: Model "decides" to read a file
            # In a full agent, the model would output a tool call here.
            # For this example, we manually trigger the 'list_directory' tool.
            result = await session.call_tool("list_files", arguments={"path": "."})
            context = result.content[0].text
            
            # 3. Pass that real-world data into Qwen
            prompt = f"Based on these files: {context}\n\nExplain the project structure:"
            inputs = tokenizer(prompt, return_tensors="pt")
            
            print("\n--- Qwen is analyzing your project ---")
            outputs = model.generate(**inputs, max_new_tokens=150)
            print(tokenizer.decode(outputs[0], skip_special_tokens=True))

# Run the async loop
# asyncio.run(run_mcp_magic()) # Use this if in a .py file
await run_agentic_qwen() # Use this in a Jupyter cell

✅ MCP Connected. Tools available: ['list_files', 'list_kernels', 'use_notebook', 'list_notebooks', 'restart_notebook', 'unuse_notebook', 'read_notebook', 'insert_cell', 'overwrite_cell_source', 'edit_cell_source', 'execute_cell', 'insert_execute_code_cell', 'read_cell', 'delete_cell', 'move_cell', 'execute_code', 'connect_to_jupyter']

--- Qwen is analyzing your project ---
Based on these files: Showing 0-1 of 1 files

Path	Type	Size	Last_Modified
.	error		Error: Connection error to http://localhost:8888/api/contents/: HTTPConnectionPool(host='localhost', port=8888): Max retries exceeded with url: /api/contents/?type=directory&content=1 (Caused by NewConnectionError("HTTPConnection(host='localhost', port=8888): Failed to establish a new connection: [Errno 111] Connection refused"))

Explain the project structure: What are the main components of the project? How are they organized?

The error message shows that the connection to the API endpoint is refused. The error message says that t